In [1]:
import requests
from urllib.request import urlretrieve
import xmltodict

from collections import Counter
from dataclasses import dataclass, field
from typing import Literal
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import arff
# import openml
from tempfile import NamedTemporaryFile

# Helpers

In [2]:
def download_and_parse(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    return xmltodict.parse(response.content)

def download_to_temp_file(
    url: str,
    suffix: str = "",
    chunk_size: int = 8192,
) -> str:
    """
    Download a URL to a temporary file.

    Returns
    -------
    str
        Path to the downloaded file.
    """
    with requests.get(url, stream=True) as response:
        response.raise_for_status()

        with NamedTemporaryFile(
            suffix=suffix,
            delete=False,
        ) as tmp:
            for chunk in response.iter_content(chunk_size=chunk_size):
                tmp.write(chunk)

    return tmp.name

# Python Parser

In [4]:
# ============================================================================
# Dataset metadata
# ============================================================================

@dataclass(slots=True, frozen=True)
class DatasetDownloadInfo:
    file_path: str
    default_target_attribute: str | None


# ============================================================================
# Feature models
# ============================================================================

@dataclass(slots=True)
class Feature:
    index: int
    name: str
    data_type: str

    nominal_values: list[str] = field(default_factory=list)

    is_target: bool = False
    is_ignore: bool = False
    is_row_identifier: bool = False

    number_of_distinct_values: int | None = None
    number_of_unique_values: int | None = None
    number_of_missing_values: int | None = None
    number_of_integer_values: int | None = None
    number_of_real_values: int | None = None
    number_of_nominal_values: int | None = None
    number_of_values: int | None = None

    maximum_value: float | None = None
    minimum_value: float | None = None
    mean_value: float | None = None
    standard_deviation: float | None = None

    class_distribution: str | None = None

    def __str__(self) -> str:
        return f"{self.index} - {self.name}"


@dataclass(slots=True)
class DataFeature:
    did: int | None = None
    evaluation_engine_id: int | None = None
    features: list[Feature] = field(default_factory=list)
    error: str | None = None

    def feature_map(
        self,
        *,
        sorted_names: bool = False,
    ) -> dict[str, Feature]:
        result = {f.name: f for f in self.features}
        return dict(sorted(result.items())) if sorted_names else result


# ============================================================================
# Constants
# ============================================================================

_NUMERIC_TYPES = frozenset({"NUMERIC", "REAL", "INTEGER"})


_OML_BOOL_FIELDS = (
    "is_target",
    "is_ignore",
    "is_row_identifier",
)

_OML_INT_FIELDS = (
    "number_of_missing_values",
    "number_of_distinct_values",
    "number_of_unique_values",
    "number_of_integer_values",
    "number_of_real_values",
    "number_of_nominal_values",
    "number_of_values",
)

_OML_FLOAT_FIELDS = (
    "maximum_value",
    "minimum_value",
    "mean_value",
    "standard_deviation",
)


# ============================================================================
# Dataset retrieval 
# ============================================================================

def get_data_and_meta_information_from_did(
    did: int,
    dataset_type: Literal["arff", "parquet"] = "arff",
) -> DatasetDownloadInfo:
    dataset_type = dataset_type.lower()

    if dataset_type not in {"arff", "parquet"}:
        raise ValueError("dataset_type must be 'arff' or 'parquet'")

    metadata = download_and_parse(
        f"https://www.openml.org/api/v1/xml/data/{did}"
    )["oml:data_set_description"]

    url_key = "oml:url" if dataset_type == "arff" else "oml:parquet_url"

    return DatasetDownloadInfo(
        file_path=download_to_temp_file(
            metadata[url_key],
            suffix=f".{dataset_type}",
        ),
        default_target_attribute=metadata.get(
            "oml:default_target_attribute"
        ),
    )


# ============================================================================
# ARFF type handling
# ============================================================================

def _liac_type(type_spec):
    if isinstance(type_spec, list):
        return "nominal", tuple(type_spec)

    normalized = type_spec.upper()

    if normalized in _NUMERIC_TYPES:
        return "numeric", None

    return {
        "STRING": ("string", None),
        "DATE": ("date", None),
    }.get(normalized, (normalized.lower(), None))


def _normalize_target_names(target: str | list[str] | None) -> set[str]:
    if target is None:
        return set()

    if isinstance(target, str):
        return {t.strip() for t in target.split(",") if t.strip()}

    return {t.strip() for t in target if t and t.strip()}


# ============================================================================
# Feature extraction
# ============================================================================

def _fill_numeric(col, feat: Feature) -> None:
    feat.number_of_values = len(col)

    present = np.asarray(
        [v for v in col if v is not None],
        dtype=np.float64,
    )

    feat.number_of_missing_values = len(col) - present.size

    if present.size == 0:
        return

    int_mask = present == np.floor(present)

    feat.number_of_integer_values = int(int_mask.sum())
    feat.number_of_real_values = int((~int_mask).sum())

    feat.maximum_value = float(np.max(present))
    feat.minimum_value = float(np.min(present))
    feat.mean_value = float(np.mean(present))
    feat.standard_deviation = float(np.std(present, ddof=0))

    unique_vals, counts = np.unique(
        present,
        return_counts=True,
    )

    feat.number_of_distinct_values = len(unique_vals)
    feat.number_of_unique_values = int(np.sum(counts == 1))


def _fill_nominal(col, feat: Feature, schema_values) -> None:
    feat.nominal_values = sorted(schema_values)
    feat.number_of_values = len(col)

    counts = Counter(
        v for v in col if v is not None
    )

    present = sum(counts.values())

    feat.number_of_missing_values = len(col) - present
    feat.number_of_nominal_values = present
    feat.number_of_distinct_values = len(counts)
    feat.number_of_unique_values = sum(c == 1 for c in counts.values())

    feat.class_distribution = ",".join(
        f"{k}:{v}" for k, v in sorted(counts.items())
    )


# ============================================================================
# MAIN LOADER 
# ============================================================================

def load_arff_features(
    dataset: DatasetDownloadInfo,
    *,
    did: int | None = None,
    evaluation_engine_id: int | None = None,
) -> DataFeature:
    try:
        with open(
            dataset.file_path,
            "r",
            encoding="utf-8",
            errors="replace",
        ) as f:
            data = arff.load(f)

    except Exception as exc:
        return DataFeature(
            did=did,
            evaluation_engine_id=evaluation_engine_id,
            error=str(exc),
        )

    attributes = data["attributes"]
    rows = data["data"]

    target_names = _normalize_target_names(
        dataset.default_target_attribute
    )

    if rows:
        columns = list(zip(*rows))
    else:
        columns = [tuple() for _ in attributes]

    features: list[Feature] = []

    for idx, ((name, type_spec), col) in enumerate(
        zip(attributes, columns)
    ):
        type_name, type_range = _liac_type(type_spec)

        feat = Feature(
            index=idx,
            name=name,
            data_type=type_name,
            is_target=name in target_names,
        )

        if type_name == "numeric":
            _fill_numeric(col, feat)

        elif type_name == "nominal":
            _fill_nominal(col, feat, type_range)

        else:
            feat.number_of_values = len(col)

        features.append(feat)

    return DataFeature(
        did=did,
        evaluation_engine_id=evaluation_engine_id,
        features=features,
    )


# ============================================================================
# SERIALIZATION 
# ============================================================================

def feature_to_oml_dict(feat: Feature) -> dict:
    result = {
        "oml:index": str(feat.index),
        "oml:name": feat.name,
        "oml:data_type": feat.data_type,
    }

    if feat.nominal_values:
        result["oml:nominal_value"] = feat.nominal_values

    for attr in _OML_BOOL_FIELDS:
        v = getattr(feat, attr)
        if v is not None:
            result[f"oml:{attr}"] = str(v).lower()

    for attr in (*_OML_INT_FIELDS, *_OML_FLOAT_FIELDS):
        v = getattr(feat, attr)
        if v is not None:
            result[f"oml:{attr}"] = str(v)

    if feat.class_distribution:
        result["oml:class_distribution"] = feat.class_distribution

    return result


def features_to_oml_dict(data_feature: DataFeature) -> dict:
    return {
        "oml:data_features": {
            "@xmlns:oml": "http://openml.org/openml",
            "oml:feature": [
                feature_to_oml_dict(f)
                for f in data_feature.features
            ],
        }
    }


def features_to_xml(
    data_feature: DataFeature,
    *,
    pretty: bool = True,
) -> str:
    return xmltodict.unparse(
        features_to_oml_dict(data_feature),
        pretty=pretty,
    )


def parse_features_xml(xml_str: str) -> dict:
    return xmltodict.parse(
        xml_str,
        force_list=("oml:feature", "oml:nominal_value"),
    )

In [5]:
data_id = 47246
arff_local_features = load_arff_features(
            get_data_and_meta_information_from_did(data_id),
            did=data_id,
        )

xml_local_features = parse_features_xml(
    features_to_xml(
        arff_local_features
    )
)



# Server features
- only for sanity check

In [6]:
data_id = 47246

In [7]:
features_url = lambda data_id:f"https://www.openml.org/api/v1/data/features/{data_id}"
server_features = lambda data_id:download_and_parse(features_url(data_id))

## Get diffs between versions

In [8]:
def get_diff_between_server_and_local_for_did(
    server: dict,
    local: DataFeature,
) -> None:
    server_features = server["oml:data_features"]["oml:feature"]

    local_features = {
        f.index: feature_to_oml_dict(f)
        for f in local.features
    }

    for s in server_features:
        idx = int(s["oml:index"])
        l = local_features.get(idx)

        if l is None:
            print(f"feature {idx}: missing locally")
            continue

        if s == l:
            continue

        print(f"feature {idx}:")

        for k in sorted(set(s) | set(l)):
            # if s.get(k) != l.get(k):
            print(f"  {k}: server={s.get(k)!r}  local={l.get(k)!r}")

In [11]:
download_info = get_data_and_meta_information_from_did(did = 47246)

In [12]:
get_diff_between_server_and_local_for_did(
    server=server_features(data_id=data_id),
    local=arff_local_features,
)

feature 0:
  oml:data_type: server='string'  local='string'
  oml:index: server='0'  local='0'
  oml:is_ignore: server='false'  local='false'
  oml:is_row_identifier: server='true'  local='false'
  oml:is_target: server='false'  local='false'
  oml:name: server='segment_id'  local='segment_id'
  oml:number_of_missing_values: server='0'  local=None
  oml:number_of_values: server=None  local='3456'
feature 1:
  oml:data_type: server='string'  local='string'
  oml:index: server='1'  local='1'
  oml:is_ignore: server='false'  local='false'
  oml:is_row_identifier: server='false'  local='false'
  oml:is_target: server='false'  local='false'
  oml:name: server='clip_id'  local='clip_id'
  oml:number_of_missing_values: server='0'  local=None
  oml:number_of_values: server=None  local='3456'
feature 2:
  oml:data_type: server='string'  local='string'
  oml:index: server='2'  local='2'
  oml:is_ignore: server='false'  local='false'
  oml:is_row_identifier: server='false'  local='false'
  oml:is

# Server parser - Java parser output

In [1]:
import os
import requests
from xml.etree import ElementTree
from urllib.request import urlretrieve
import xmltodict

from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import arff


In [2]:
import openml

In [3]:
def get_ds_and_return_arff_path_and_target(did:int):
    openml.datasets.get_dataset(did).get_data()
    # return Path(openml.config.get_cache_directory())/f"datasets/{did}/dataset_{did}.arff"
    d = openml.datasets.get_dataset(did)
    with open("test.arff", "wb") as f:
        f.write(requests.get(d.url).content)

    target = d.default_target_attribute
    return "test.arff", target

In [4]:
get_ds_and_return_arff_path_and_target(47246)

('test.arff', 'translated_text')

In [28]:
!curl https://www.openml.org/api/v1/json/data/47246 | jq ".data_set_description.default_target_attribute"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3047  100  3047    0     0  10466      0 --:--:-- --:--:-- --:--:-- 10470
"translated_text"


In [6]:
features = lambda id:f"https://www.openml.org/api/v1/data/features/{id}"

In [7]:
def download_and_parse(url):
    response = requests.get(url)
    return xmltodict.parse(response.content)

# Server parser - Python

In [8]:

@dataclass
class Feature:
    index: Optional[int] = None
    name: Optional[str] = None
    data_type: Optional[str] = None
    nominal_values: List[str] = field(default_factory=list)
    is_target: Optional[bool] = None
    is_ignore: Optional[bool] = None
    is_row_identifier: Optional[bool] = None
    number_of_distinct_values: Optional[int] = None
    number_of_unique_values: Optional[int] = None
    number_of_missing_values: Optional[int] = None
    number_of_integer_values: Optional[int] = None
    number_of_real_values: Optional[int] = None
    number_of_nominal_values: Optional[int] = None
    number_of_values: Optional[int] = None
    maximum_value: Optional[float] = None
    minimum_value: Optional[float] = None
    mean_value: Optional[float] = None
    standard_deviation: Optional[float] = None
    class_distribution: Optional[str] = None
 
    def __str__(self) -> str:
        return f"{self.index} - {self.name}"
 p
 p

In [9]:
@dataclass
class DataFeature:
    did: Optional[int] = None
    evaluation_engine_id: Optional[int] = None
    features: List[Feature] = field(default_factory=list)
    error: Optional[str] = None

    def get_features_as_map(self) -> Dict[str, Feature]:
        """Sorted by name — mirrors Java's TreeMap-based getFeaturesAsMap()."""
        return dict(sorted({f.name: f for f in self.features}.items()))

    def get_feature_map(self) -> Dict[str, Feature]:
        """Unsorted — mirrors Java's HashMap-based getFeatureMap()."""
        return {f.name: f for f in self.features}


# ─────────────────────────────────────────────────────────────────────────────
# Internal helpers
# ─────────────────────────────────────────────────────────────────────────────

def _fill_numeric(col, feat) -> None:
    """col is a list of values; None marks missing (liac-arff decodes '?' as None)."""
    feat.number_of_values = len(col)

    present = np.array([v for v in col if v is not None], dtype=float)
    feat.number_of_missing_values = len(col) - present.size

    if present.size == 0:
        return

    int_mask = present == np.floor(present)
    feat.number_of_integer_values = int(int_mask.sum())
    feat.number_of_real_values = int((~int_mask).sum())

    feat.maximum_value = float(present.max())
    feat.minimum_value = float(present.min())
    feat.mean_value = float(present.mean())
    # population std — ddof=0 matches OpenML's weka-based computation
    feat.standard_deviation = float(present.std(ddof=0))

    unique_vals = np.unique(present)
    feat.number_of_distinct_values = len(unique_vals)

    counts = Counter(present.tolist())
    feat.number_of_unique_values = sum(1 for c in counts.values() if c == 1)


def _fill_nominal(col, feat, schema_values) -> None:
    """col is a list of strings; None marks missing."""
    feat.nominal_values = sorted(schema_values)
    feat.number_of_values = len(col)

    present = [v for v in col if v is not None]
    feat.number_of_missing_values = len(col) - len(present)
    feat.number_of_nominal_values = len(present)

    counts = Counter(present)
    feat.number_of_distinct_values = len(counts)
    feat.number_of_unique_values = sum(1 for c in counts.values() if c == 1)
    feat.class_distribution = ",".join(
        f"{label}:{cnt}" for label, cnt in sorted(counts.items())
    )


# ─────────────────────────────────────────────────────────────────────────────
# Public loader  (liac-arff based)
# ─────────────────────────────────────────────────────────────────────────────

def _liac_type(type_spec):
    """Map liac-arff's attribute type spec to scipy-style type names."""
    if isinstance(type_spec, list):
        return "nominal", tuple(type_spec)
    u = type_spec.upper()
    if u in ("NUMERIC", "REAL", "INTEGER"):
        return "numeric", None
    if u == "STRING":
        return "string", None
    if u == "DATE":
        return "date", None
    return u.lower(), None


def _normalize_target_names(target):
    """OpenML's default_target_attribute may be a single name or a comma-separated
    string for multi-target datasets. Normalize to a set of stripped names."""
    if target is None:
        return set()
    if isinstance(target, str):
        return {t.strip() for t in target.split(",") if t.strip()}
    return {t.strip() for t in target if t and t.strip()}


def load_arff_features(
    arff_path: Path | str,
    did: Optional[int] = None,
    evaluation_engine_id: Optional[int] = None,
    target: Optional[str | List[str]] = None,
) -> DataFeature:
    """Load an ARFF file via liac-arff and compute per-feature statistics."""
    try:
        # liac-arff's decoder matches string regex tokens line-by-line, so the
        # file must be opened in text mode. errors="replace" keeps legacy
        # non-UTF-8 OpenML datasets parseable instead of crashing on decode.
        with open(arff_path, "r", encoding="utf-8", errors="replace") as f:
            dataset = arff.load(f)
    except Exception as exc:
        return DataFeature(did=did, evaluation_engine_id=evaluation_engine_id, error=str(exc))

    target_names = _normalize_target_names(target)
    attributes = dataset["attributes"]
    rows = dataset["data"]

    features: List[Feature] = []
    for idx, (name, type_spec) in enumerate(attributes):
        type_name, type_range = _liac_type(type_spec)
        col = [row[idx] for row in rows]
        feat = Feature(index=idx, name=name, data_type=type_name)
        if name in target_names:
            feat.is_target = True

        if type_name == "numeric":
            _fill_numeric(col, feat)
        elif type_name == "nominal":
            _fill_nominal(col, feat, type_range)
        else:
            # string / date / relational — just count rows
            feat.number_of_values = len(col)

        features.append(feat)

    return DataFeature(did=did, evaluation_engine_id=evaluation_engine_id, features=features)


In [10]:
_OML_BOOL_FIELDS = ("is_target", "is_ignore", "is_row_identifier")
_OML_INT_FIELDS = (
    "number_of_missing_values",
    "number_of_distinct_values",
    "number_of_unique_values",
    "number_of_integer_values",
    "number_of_real_values",
    "number_of_nominal_values",
    "number_of_values",
)
_OML_FLOAT_FIELDS = (
    "maximum_value",
    "minimum_value",
    "mean_value",
    "standard_deviation",
)


def feature_to_oml_dict(feat: Feature) -> dict:
    d: dict = {
        "oml:index": str(feat.index),
        "oml:name": feat.name,
        "oml:data_type": feat.data_type,
    }
    if feat.data_type == "nominal" and feat.nominal_values:
        d["oml:nominal_value"] = list(feat.nominal_values)
    for attr in _OML_BOOL_FIELDS:
        v = getattr(feat, attr)
        if v is not None:
            d[f"oml:{attr}"] = "true" if v else "false"
    for attr in _OML_INT_FIELDS + _OML_FLOAT_FIELDS:
        v = getattr(feat, attr)
        if v is not None:
            d[f"oml:{attr}"] = str(v)
    if feat.class_distribution is not None:
        d["oml:class_distribution"] = feat.class_distribution
    return d


def features_to_oml_dict(data_feature: DataFeature) -> dict:
    return {
        "oml:data_features": {
            "@xmlns:oml": "http://openml.org/openml",
            "oml:feature": [feature_to_oml_dict(f) for f in data_feature.features],
        }
    }


def features_to_xml(data_feature: DataFeature, pretty: bool = True) -> str:
    return xmltodict.unparse(features_to_oml_dict(data_feature), pretty=pretty)


def parse_features_xml(xml_str: str) -> dict:
    """xmltodict.parse with force_list for oml:feature and oml:nominal_value,
    so round-trips through features_to_xml are stable."""
    return xmltodict.parse(xml_str, force_list=("oml:feature", "oml:nominal_value"))

In [11]:
# test_arff_path = Path("data/dataset_10_lymph.arff")
# load_arff_features(test_arff_path).features[11]
# parse_features_xml(features_to_xml(load_arff_features(test_arff_path)))

# Get diffs between versions

In [12]:

def get_diff_between_server_and_local_for_did(server, local):
    for s, l in zip(server["oml:data_features"]["oml:feature"],
                    local["oml:data_features"]["oml:feature"]):
        if s != l:
            print(f"feature {s['oml:index']}:")
            for k in set(s) | set(l):
                # if s.get(k) != l.get(k):
                print(f"  {k}: server={s.get(k)!r}  local={l.get(k)!r}")

# Testing

In [13]:
data_id = 47246

In [14]:
server = download_and_parse(features(data_id))
arff_path, target_name = get_ds_and_return_arff_path_and_target(data_id)
local  = parse_features_xml(features_to_xml(load_arff_features(arff_path, target=target_name)))

In [16]:
get_diff_between_server_and_local_for_did(server, local)

feature 0:
  oml:is_target: server='false'  local=None
  oml:index: server='0'  local='0'
  oml:data_type: server='string'  local='string'
  oml:number_of_missing_values: server='0'  local=None
  oml:is_ignore: server='false'  local=None
  oml:name: server='segment_id'  local='segment_id'
  oml:is_row_identifier: server='true'  local=None
  oml:number_of_values: server=None  local='3456'
feature 1:
  oml:is_target: server='false'  local=None
  oml:index: server='1'  local='1'
  oml:data_type: server='string'  local='string'
  oml:number_of_missing_values: server='0'  local=None
  oml:is_ignore: server='false'  local=None
  oml:name: server='clip_id'  local='clip_id'
  oml:is_row_identifier: server='false'  local=None
  oml:number_of_values: server=None  local='3456'
feature 2:
  oml:is_target: server='false'  local=None
  oml:index: server='2'  local='2'
  oml:data_type: server='string'  local='string'
  oml:number_of_missing_values: server='0'  local=None
  oml:is_ignore: server='fals